# Fine-tune PhoBERT on Colab

Trains `vinai/phobert-base-v2` on `data/labeled/news_labeled.csv` (6 labels) with 3
seeds, reports validation macro-F1 (mean ± std) and uploads the best model to the
Hugging Face Hub. The blind test set is **not** used here.

Before running:
- Runtime -> Change runtime type -> **GPU** (T4 is enough).
- Colab Secrets (key icon): add `HF_TOKEN`, a Hugging Face token with **write**
  access, and allow this notebook to read it.
- Check `HF_REPO_ID` in `src/finetune/config.py` (your Hugging Face username).

In [ ]:
!git clone https://github.com/maidoanthanhthuy/financial-news-event-classifier.git
%cd financial-news-event-classifier
!pip install -q underthesea -r requirements-finetune.txt

In [ ]:
import torch

print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## Train with 3 seeds
The train/validation split is fixed (85/15 by label, seed 42); the seed changes model initialization and data order.

In [ ]:
SEEDS = [42, 43, 44]
for seed in SEEDS:
    !python -m src.finetune.train --seed {seed}

## Validation results

In [ ]:
import json

import pandas as pd

from src.finetune.config import OUTPUT_DIR
from src.finetune.train import summarize_seeds

results = [json.load(open(f"{OUTPUT_DIR}_seed{s}/metrics.json")) for s in SEEDS]
display(pd.DataFrame(results))
summary = summarize_seeds(results)
print(f"validation macro-F1: {summary['mean']:.3f} ± {summary['std']:.3f} (best seed {summary['best_seed']})")

## Upload the best model to the Hugging Face Hub

In [ ]:
from google.colab import userdata

from src.finetune.config import HF_REPO_ID
from src.finetune.train import push_to_hub

push_to_hub(f"{OUTPUT_DIR}_seed{summary['best_seed']}", HF_REPO_ID, userdata.get("HF_TOKEN"))
print(f"https://huggingface.co/{HF_REPO_ID}")

## Check: load from the Hub and predict one item

In [ ]:
from src.finetune.predict import predict_phobert

predict_phobert("FPT báo lãi quý III tăng 20%", "Doanh thu đạt 15.000 tỷ đồng.")

## Blind test set: score every seed

Run after training (the seed models must be in `OUTPUT_DIR`). Predicts
`data/labeled/test_blind.csv` with each seed model and prints accuracy, macro-F1,
per-label scores and mean ± std over seeds.

In [ ]:
import glob

import pandas as pd

from src.classify.config import LABELS
from src.classify.evaluate import join_predictions, per_label_scores, summary_scores
from src.finetune.config import OUTPUT_DIR, TEST_CSV
from src.finetune.predict import predict_frame
from src.finetune.train import summarize_seeds

labels = list(LABELS)
truth = pd.read_csv(TEST_CSV)
test_results = []
for model_dir in sorted(glob.glob(f"{OUTPUT_DIR}_seed*")):
    seed = int(model_dir.rsplit("seed", 1)[1])
    df = join_predictions(truth, predict_frame(truth, model_dir), "label")
    scores = summary_scores(df["human_label"], df["pred"], labels)
    test_results.append({"seed": seed} | scores)
    print(f"seed {seed}: accuracy {scores['accuracy']:.3f}, macro-F1 {scores['macro_f1']:.3f}")
    display(per_label_scores(df["human_label"], df["pred"], labels).round(3))

display(pd.DataFrame(test_results))
for metric in ["accuracy", "macro_f1"]:
    s = summarize_seeds(test_results, metric)
    print(f"test {metric}: {s['mean']:.3f} ± {s['std']:.3f}")